![JohnSnowLabs](https://sparknlp.org/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/transformers/Choosing_Your_Engine_ONNX_vs_OpenVINO_in_Spark_NLP.ipynb)

# Choosing Your Inference Engine: ONNX vs OpenVINO in Spark NLP 🚀

This notebook walks you through the `engine` parameter introduced in Spark NLP, which lets you **choose which deep learning backend** is used when downloading pretrained models.

Spark NLP supports multiple inference backends:
- **`tensorflow`** — the original TensorFlow backend (older models)
- **`onnx`** — high-performance cross-platform runtime via [ONNX Runtime](https://onnxruntime.ai/) *(default since Spark NLP 5.0.0)*
- **`openvino`** — Intel-optimized runtime via [OpenVINO™ Toolkit](https://www.intel.com/content/www/us/en/developer/tools/openvino-toolkit/overview.html) *(since Spark NLP 5.4.0)*

The engine parameter is exposed through:
- **`pretrainedEngine(name, lang, engine=...)`** — download a pretrained model with a specific engine backend

Let's keep in mind a few things before we start 😊
- The engine you pick **changes the actual binary file downloaded** — ONNX models ship `.onnx` weights while OpenVINO models ship `.xml`/`.bin` weights. You can verify this with `ls` directly in the Spark NLP cache folder.
- All engines produce the **same results** for the same model — the difference is purely about runtime performance characteristics and hardware compatibility.
- ONNX is the default and works on all hardware. OpenVINO is optimized for Intel CPUs/GPUs/NPUs and can give significant speedups on those platforms.

## Table of Contents

1. [Install Dependencies](#1-install-dependencies)
2. [Start Spark NLP](#2-start-spark-nlp)
3. [Download the Same Model with Different Engines](#4-download-the-same-model-with-different-engines)
4. [When to Use Which Engine](#8-when-to-use-which-engine)

## 1. Install Dependencies



In [1]:
!pip install -q pyspark==3.5.4 spark-nlp==6.4.1rc1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 772.6/772.6 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 13.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.4 which is incompatible.


## 2. Start Spark NLP

Let's start Spark with Spark NLP included via our simple `start()` function.

In [1]:
from pyspark.sql import SparkSession
import sparknlp

spark = SparkSession.builder \
    .appName("Spark NLP") \
    .config("spark.driver.memory", "16G") \
    .config("spark.kryoserializer.buffer.max", "2000M") \
    .config("spark.jars", "/content/sparknlp.jar") \
    .getOrCreate()

print("Spark NLP version  :", sparknlp.version())
print("Apache Spark version:", spark.version)

Spark NLP version  : 6.4.1-rc1
Apache Spark version: 3.5.4


In [ ]:
import sparknlp

spark = sparknlp.start()

print("Spark NLP version  :", sparknlp.version())
print("Apache Spark version:", spark.version)

## 3. Download the Same Model with Different Engines

Let's download **`distilbert_base_cased`** using both the `onnx` and `openvino` engines.


In [2]:
from sparknlp.annotator import *

model_onnx = DistilBertEmbeddings.pretrainedEngine("distilbert_base_cased", "en", engine="onnx")

print(f"\n✅ Loaded! Engine reported by model: {model_onnx.getEngine()}")

distilbert_base_cased download started this may take some time.
Approximate size to download 232.5 MB
[OK!]

✅ Loaded! Engine reported by model: onnx


In [3]:
model_openvino = DistilBertEmbeddings.pretrainedEngine("distilbert_base_cased", "en", engine="openvino")

print(f"\n✅ Loaded! Engine reported by model: {model_openvino.getEngine()}")

distilbert_base_cased download started this may take some time.
Approximate size to download 232.5 MB
[OK!]

✅ Loaded! Engine reported by model: openvino


## 4. When to Use Which Engine

### Quick decision guide

| Scenario | Recommended engine |
|---|---|
| Running on a mixed or unknown hardware cluster | `onnx` (default) |
| Running on Intel CPUs (Xeon, Core) | `openvino` |
| Running on Intel integrated GPU or Arc GPU | `openvino` |
| Running on Intel NPU (Core Ultra) | `openvino` |
| Running on NVIDIA GPU | `onnx` (with CUDA EP) |
| Reproducing results from old Spark NLP models | `tensorflow` |
| Maximum portability and ecosystem compatibility | `onnx` |

### Performance notes

- **OpenVINO** typically gives **1.5×–4× throughput improvement** over ONNX on Intel CPUs due to model-level graph optimizations and hardware-specific kernel fusion.
- **ONNX** is the safest choice for heterogeneous clusters (a mix of Intel, AMD, ARM workers) since it runs everywhere.
- Both `onnx` and `openvino` are significantly faster than `tensorflow` for inference in Spark NLP.

